In [ ]:
# 忽视警告，这个库是内置的，不需要安装
from pathlib import Path
import importlib.util
import sys
import warnings

import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

warnings.filterwarnings("ignore")

support_path_candidates = [
    Path("cuda_training_support.py"),
    Path("..") / "cuda_training_support.py",
]
SUPPORT_PATH = next((path.resolve() for path in support_path_candidates if path.exists()), None)
if SUPPORT_PATH is None:
    raise FileNotFoundError("找不到 cuda_training_support.py")

support_spec = importlib.util.spec_from_file_location("cuda_training_support", SUPPORT_PATH)
cuda_training_support = importlib.util.module_from_spec(support_spec)
sys.modules["cuda_training_support"] = cuda_training_support
support_spec.loader.exec_module(cuda_training_support)

NOTEBOOK_CONFIG = cuda_training_support.build_notebook_run_config()
DEFAULT_TARGET_COLUMN = cuda_training_support.DEFAULT_TARGET_COLUMN

# 兼容从项目根目录或 Notebook 所在目录启动内核的两种情况。
DATA_PATH = cuda_training_support.resolve_data_path(start_dir=Path.cwd())

print(f"训练模式: {NOTEBOOK_CONFIG.run_mode}")
print(f"数据文件: {DATA_PATH}")
print(f"Smoke sample size: {NOTEBOOK_CONFIG.sample_size}")
print(f"BayesSearch n_iter: {NOTEBOOK_CONFIG.bayes_n_iter}")
print(f"CV folds: {NOTEBOOK_CONFIG.cv_folds}")


In [ ]:
import pandas as pd

random_seed = NOTEBOOK_CONFIG.random_seed

data = cuda_training_support.load_training_dataframe(
    data_path=DATA_PATH,
    random_seed=random_seed,
    run_mode=NOTEBOOK_CONFIG.run_mode,
    sample_size=NOTEBOOK_CONFIG.sample_size,
    target_column=DEFAULT_TARGET_COLUMN,
)

# 显示前几行数据，以确认数据已正确加载
(data.head(10))


In [ ]:
import pandas as pd
data['RETENTION_TIME'] = pd.to_numeric(data['RETENTION_TIME'], errors='coerce').astype('float64')
# 假设 df 是你的 DataFrame
print("转换前 RETENTION_TIME 的示例值：")
print(data['RETENTION_TIME'].head())

# 检查非数值数据（如字符串、空值等）
non_numeric = pd.to_numeric(data['RETENTION_TIME'], errors='coerce').isna()
print("\n非数值数据数量：", non_numeric.sum())
print("非数值数据示例：")
print(data[non_numeric]['RETENTION_TIME'].unique())  # 查看具体非数值内容

In [ ]:
print("样本总数:", len(data))
print("标签分布:")
data[DEFAULT_TARGET_COLUMN].value_counts()


In [ ]:
prepared = cuda_training_support.prepare_lightgbm_training_data(
    data,
    target_column=DEFAULT_TARGET_COLUMN,
    random_state=NOTEBOOK_CONFIG.random_seed,
    test_size=NOTEBOOK_CONFIG.test_size,
)

X_train = prepared["X_train"]
X_test = prepared["X_test"]
y_train = prepared["y_train"]
y_test = prepared["y_test"]
scaler = prepared["scaler"]

print("过采样方法:", "notebook_smote")
print("过采样后的训练集形状:", X_train.shape)
print("测试集形状:", X_test.shape)
print("过采样后的标签分布:", pd.Series(y_train).value_counts())


## LightGBM（CUDA-only）

- 训练设备固定为 `cuda`，禁止 CPU fallback。
- Notebook 默认使用 `smoke` 模式做轻量验证；设置环境变量 `LGBM_NOTEBOOK_RUN_MODE=full` 可切换为全量训练。
- 训练前会执行一次 CUDA 预检；如果当前 `lightgbm` 不是带 CUDA 的构建，会直接报错并停止。


In [ ]:
import sys

# 如果内核从项目根目录启动，先移除本地 lightgbm 目录对官方包导入的遮蔽。
current_dir = Path.cwd().resolve()
if (current_dir / "lightgbm").is_dir():
    sys.path = [
        path
        for path in sys.path
        if Path(path or current_dir).resolve() != current_dir
    ]

import lightgbm as lgb
from skopt import BayesSearchCV

lgbm_version = cuda_training_support.validate_lightgbm_cuda_build(
    random_state=NOTEBOOK_CONFIG.random_seed,
)
print("LightGBM CUDA preflight:", lgbm_version)

# 定义 LightGBM 模型，设备被强制锁定为 CUDA。
lgb_model = cuda_training_support.build_lgbm_classifier(
    random_state=NOTEBOOK_CONFIG.random_seed,
    model_n_jobs=NOTEBOOK_CONFIG.model_n_jobs,
)

# 定义贝叶斯搜索空间。smoke 模式使用更小的搜索范围，避免误触全量计算。
search_spaces = cuda_training_support.get_lgbm_search_spaces(NOTEBOOK_CONFIG.run_mode)

# 初始化贝叶斯搜索
bayes_search = BayesSearchCV(
    estimator=lgb_model,
    search_spaces=search_spaces,
    n_iter=NOTEBOOK_CONFIG.bayes_n_iter,
    cv=NOTEBOOK_CONFIG.cv_folds,
    scoring="roc_auc",
    n_jobs=NOTEBOOK_CONFIG.search_n_jobs,
    verbose=1,
    random_state=NOTEBOOK_CONFIG.random_seed,
)

# 在训练集上执行贝叶斯搜索
bayes_search.fit(X_train, y_train)

# 输出最佳参数
print("最佳参数组合:", bayes_search.best_params_)
print("最佳验证集AUC:", bayes_search.best_score_)

# 使用最佳模型预测测试集
y_pred_proba = bayes_search.best_estimator_.predict_proba(X_test)[:, 1]
test_auc = roc_auc_score(y_test, y_pred_proba)
print("测试集AUC:", test_auc)

# 可选：也输出测试集的准确率和 F1 分数
y_pred = bayes_search.best_estimator_.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred)
test_f1 = f1_score(y_test, y_pred)
print("测试集准确率:", test_accuracy)
print("测试集F1分数:", test_f1)


In [ ]:
# 使用最佳参数训练模型
best_lgb = bayes_search.best_estimator_

# ============ 训练集评估 ============
y_train_pred = best_lgb.predict(X_train)
y_train_proba = best_lgb.predict_proba(X_train)[:, 1]

# 计算训练集指标
train_metrics = {
    'AUC': roc_auc_score(y_train, y_train_proba),
    'Accuracy': accuracy_score(y_train, y_train_pred),
    'Balanced Accuracy': balanced_accuracy_score(y_train, y_train_pred),
    'Precision': precision_score(y_train, y_train_pred),
    'Recall': recall_score(y_train, y_train_pred),
    'F1': f1_score(y_train, y_train_pred)
}

# 计算Specificity
tn, fp, fn, tp = confusion_matrix(y_train, y_train_pred).ravel()
train_metrics['Specificity'] = tn / (tn + fp)

# ============ 测试集评估 ============
y_test_pred = best_lgb.predict(X_test)
y_test_proba = best_lgb.predict_proba(X_test)[:, 1]

# 计算测试集指标
test_metrics = {
    'AUC': roc_auc_score(y_test, y_test_proba),
    'Accuracy': accuracy_score(y_test, y_test_pred),
    'Balanced Accuracy': balanced_accuracy_score(y_test, y_test_pred),
    'Precision': precision_score(y_test, y_test_pred),
    'Recall': recall_score(y_test, y_test_pred),
    'F1': f1_score(y_test, y_test_pred)
}

# 计算Specificity
tn, fp, fn, tp = confusion_matrix(y_test, y_test_pred).ravel()
test_metrics['Specificity'] = tn / (tn + fp)

# ============ 打印结果 ============
print("\n=== 训练集性能 ===")
for metric, value in train_metrics.items():
    print(f"{metric:<18}: {value:.4f}")

print("\n=== 测试集性能 ===")
for metric, value in test_metrics.items():
    print(f"{metric:<18}: {value:.4f}")

# 打印混淆矩阵
print("\n测试集混淆矩阵:")
print(confusion_matrix(y_test, y_test_pred))

# 特征重要性可视化（可选）
lgb.plot_importance(best_lgb, max_num_features=20)
plt.tight_layout()
plt.show()